# Use Case 1: Compound-Centric Queries

**Question:** *Given a compound, what do we know about it — its targets, source sets, primary target, and selectivity?*

**Examples:** JQ1 (BET bromodomain probe) and Imatinib (BCR-ABL inhibitor, approved drug)

This notebook demonstrates how to query the Probes & Drugs database starting from a **compound name** and traversing to targets, sets, and selectivity metrics.

## Schema paths used

```
compound → activity → target → targettobasetarget → basetarget     (targets + potency)
compound → compoundaction → target → targettobasetarget → basetarget  (curated actions)
compound → compoundtocompoundset → compoundset → compoundsettype   (source sets)
compound → compoundbasetargetcriteria → basetarget                 (pre-computed selectivity)
```

## Setup

All queries use the shared `pd_utils` module. Import it once, then call functions by name.


In [1]:
# Import shared utilities — all query functions in one place
import sys
sys.path.insert(0, '/mnt/results')

from pd_utils import *

# Verify database connection
list_tables()

,table,n_rows
0,actiontype,46
1,activity,1191343
2,basetarget,13554
3,compound,198324
4,compoundaction,60830
5,compoundbasetargetcriteria,5310
6,compoundset,95
7,compoundsettype,9
8,compoundtargetscore,195194
9,compoundtocompoundset,337627


---
## 1a. What targets does the compound have activity against?

The `get_compound_targets()` function joins compound → activity → target → basetarget and returns the **best potency per target**, deduplicated. By default it applies three quality filters: log-scale activity types only, exact measurements only (no screening negatives), and confidence = 1 (directly measured). See the summary at the bottom for details.

In [2]:
# JQ1 targets — best potency per target
df_jq1_targets = get_compound_targets('(+)-JQ1')
print(f"JQ1: {len(df_jq1_targets)} targets")
df_jq1_targets

Raw SQL output (11 rows):
  pdid | compound_name | target_gene | target_name | best_potency | activity_types | value_types | n_measurements
  --------------------------------------------------------------------------------
  PD000031 | (+)-JQ1 | BRD4 | Bromodomain-containing pr | 9.0 | pKd,pIC50,pKi,pEC50 | median,= | 290
  PD000031 | (+)-JQ1 | BRD2 | Bromodomain-containing pr | 8.41 | pKd,pKi,pIC50 | = | 53
  PD000031 | (+)-JQ1 | BRD3 | Bromodomain-containing pr | 8.4 | pKd,pKi,pIC50 | = | 42
  PD000031 | (+)-JQ1 | BRDT | Bromodomain testis-specif | 8.4 | pIC50,pKd | median,= | 34
  PD000031 | (+)-JQ1 | CCL2 | C-C motif chemokine 2 | 8.2 | pIC50 | = | 2
  PD000031 | (+)-JQ1 | HDAC1,HDAC10,HDAC11,HDAC2 | Histone deacetylase | 6.54 | pIC50 | = | 1
  PD000031 | (+)-JQ1 | DNER | Delta and Notch-like epid | 6.3 | pIC50 | = | 1
  PD000031 | (+)-JQ1 | CREBBP | CREB-binding protein | 5.02 | pIC50,pKd | = | 3
  PD000031 | (+)-JQ1 | JAK2 | Tyrosine-protein kinase J | 5.0 | pIC50 | = | 1
  PD000

JQ1: 11 targets


,pdid,compound_name,target_gene,target_name,best_potency,activity_types,value_types,n_measurements,best_activity_type,contradiction_flag
0,PD000031,(+)-JQ1,BRD4,Bromodomain-containing protein 4,9.00,"pKd,pIC50,pKi,pEC50","median,=",290,pKd,False
1,PD000031,(+)-JQ1,BRD2,Bromodomain-containing protein 2,8.41,"pKd,pKi,pIC50",=,53,pIC50,False
2,PD000031,(+)-JQ1,BRD3,Bromodomain-containing protein 3,8.40,"pKd,pKi,pIC50",=,42,pIC50,False
3,PD000031,(+)-JQ1,BRDT,Bromodomain testis-specific protein,8.40,"pIC50,pKd","median,=",34,pIC50,False
4,PD000031,(+)-JQ1,CCL2,C-C motif chemokine 2,8.20,pIC50,=,2,pIC50,False
5,PD000031,(+)-JQ1,"HDAC1,HDAC10,HDAC11,HDAC2,HDAC3,HDAC4,HDAC5,HD...",Histone deacetylase,6.54,pIC50,=,1,pIC50,False
6,PD000031,(+)-JQ1,DNER,Delta and Notch-like epidermal growth factor-r...,6.30,pIC50,=,1,pIC50,False
7,PD000031,(+)-JQ1,CREBBP,CREB-binding protein,5.02,"pIC50,pKd",=,3,pIC50,False
8,PD000031,(+)-JQ1,JAK2,Tyrosine-protein kinase JAK2,5.00,pIC50,=,1,pIC50,False
9,PD000031,(+)-JQ1,EP300,Histone acetyltransferase p300,4.71,pIC50,=,1,pIC50,False


In [3]:
# Imatinib targets — best potency per target
df_imatinib_targets = get_compound_targets('IMATINIB')
print(f"Imatinib: {len(df_imatinib_targets)} targets")
df_imatinib_targets.head(20)

Raw SQL output (89 rows):
  pdid | compound_name | target_gene | target_name | best_potency | activity_types | value_types | n_measurements
  --------------------------------------------------------------------------------
  PD001319 | IMATINIB | ERBB2 | Receptor tyrosine-protein | 10.22 | pIC50 | = | 1
  PD001319 | IMATINIB | EGFR | Epidermal growth factor r | 9.96 | pKd,pIC50 | = | 3
  PD001319 | IMATINIB | DDR1 | Epithelial discoidin doma | 9.15 | pIC50,pKd | median,= | 9
  PD001319 | IMATINIB | ABL1 | Tyrosine-protein kinase A | 9.0 | pIC50,pKd,pKi,pPotency | median,= | 111
  PD001319 | IMATINIB | ABL1,BCR | Bcr/Abl fusion protein | 8.96 | pIC50,pKi,pEC50 | = | 52
  PD001319 | IMATINIB | PDGFRA | Platelet-derived growth f | 8.7 | pIC50,pEC50,pKd,pKi | = | 13
  PD001319 | IMATINIB | ABL2 | Tyrosine-protein kinase A | 8.22 | pKd,pIC50 | = | 14
  PD001319 | IMATINIB | BCR | Breakpoint cluster region | 8.15 | pKd | = | 1
  PD001319 | IMATINIB | CSF1R | Macrophage colony-stimula | 8.0 |

Imatinib: 89 targets


,pdid,compound_name,target_gene,target_name,best_potency,activity_types,value_types,n_measurements,best_activity_type,contradiction_flag
0,PD001319,IMATINIB,ERBB2,Receptor tyrosine-protein kinase erbB-2,10.22,pIC50,=,1,pIC50,True
1,PD001319,IMATINIB,EGFR,Epidermal growth factor receptor,9.96,"pKd,pIC50",=,3,pIC50,True
2,PD001319,IMATINIB,DDR1,Epithelial discoidin domain-containing receptor 1,9.15,"pIC50,pKd","median,=",9,pKd,False
3,PD001319,IMATINIB,ABL1,Tyrosine-protein kinase ABL1,9.00,"pIC50,pKd,pKi,pPotency","median,=",111,pKd,False
4,PD001319,IMATINIB,"ABL1,BCR",Bcr/Abl fusion protein,8.96,"pIC50,pKi,pEC50",=,52,pIC50,False
5,PD001319,IMATINIB,PDGFRA,Platelet-derived growth factor receptor alpha,8.70,"pIC50,pEC50,pKd,pKi",=,13,pIC50,False
6,PD001319,IMATINIB,ABL2,Tyrosine-protein kinase ABL2,8.22,"pKd,pIC50",=,14,pKd,False
7,PD001319,IMATINIB,BCR,Breakpoint cluster region protein,8.15,pKd,=,1,pKd,False
8,PD001319,IMATINIB,CSF1R,Macrophage colony-stimulating factor 1 receptor,8.00,"pIC50,pKd,pKi",=,8,pKd,False
9,PD001319,IMATINIB,KIT,Mast/stem cell growth factor receptor Kit,7.90,"pIC50,pKd,pEC50,pKi,pPotency",=,58,pKd,False


### Curated mechanism-of-action annotations

The `get_compound_actions()` function uses the `compoundaction` table, which provides curated annotations like *inhibitor*, *agonist*, etc., along with `primary_target` and `drug_target` flags.

In [4]:
# Curated actions for both compounds
df_jq1_actions = get_compound_actions('(+)-JQ1')
df_imatinib_actions = get_compound_actions('IMATINIB')

print(f"JQ1: {len(df_jq1_actions)} curated actions")
print(f"Imatinib: {len(df_imatinib_actions)} curated actions")
df_imatinib_actions

Raw SQL output (9 rows):
  pdid | compound_name | target_gene | action | action_type | primary_target | drug_target
  --------------------------------------------------------------------------------
  PD000031 | (+)-JQ1 | BRDT | inhibitor | negative | 0 | NULL
  PD000031 | (+)-JQ1 | BRDT | inhibitor | negative | 0 | NULL
  PD000031 | (+)-JQ1 | BRD4 | inhibitor | negative | 0 | NULL
  PD000031 | (+)-JQ1 | BRD4 | inhibitor | negative | NULL | NULL
  PD000031 | (+)-JQ1 | BRDT | inhibitor | negative | NULL | NULL
  PD000031 | (+)-JQ1 | BRD2 | inhibitor | negative | NULL | NULL
  PD000031 | (+)-JQ1 | BRD4 | inhibitor | negative | NULL | NULL
  PD000031 | (+)-JQ1 | BRD3 | inhibitor | negative | NULL | NULL
  PD000031 | (+)-JQ1 | BRDT | inhibitor | negative | NULL | NULL



Raw SQL output (24 rows):
  pdid | compound_name | target_gene | action | action_type | primary_target | drug_target
  --------------------------------------------------------------------------------
  PD001319 | IMATINIB | ABL1 | inhibitor | negative | 1 | NULL
  PD001319 | IMATINIB | DDR1 | inhibitor | negative | 0 | NULL
  PD001319 | IMATINIB | DDR2 | inhibitor | negative | 0 | NULL
  PD001319 | IMATINIB | BCR | inhibitor | negative | NULL | NULL
  PD001319 | IMATINIB | MCL1 | other | other | NULL | NULL
  PD001319 | IMATINIB | KIT | antagonist | negative | NULL | NULL
  PD001319 | IMATINIB | RET | inhibitor | negative | NULL | NULL
  PD001319 | IMATINIB | PDGFRA | antagonist | negative | NULL | NULL
  PD001319 | IMATINIB | ABL1 | inhibitor | negative | NULL | NULL
  PD001319 | IMATINIB | PDGFRB | antagonist | negative | NULL | NULL
  PD001319 | IMATINIB | NTRK1 | antagonist | negative | NULL | NULL
  PD001319 | IMATINIB | CSF1R | antagonist | negative | NULL | NULL
  PD001319 | IMA

,pdid,compound_name,target_gene,action,action_type,primary_target,drug_target
0,PD001319,IMATINIB,ABL1,inhibitor,negative,1.0,None
1,PD001319,IMATINIB,DDR1,inhibitor,negative,0.0,None
2,PD001319,IMATINIB,DDR2,inhibitor,negative,0.0,None
3,PD001319,IMATINIB,BCR,inhibitor,negative,NaN,None
4,PD001319,IMATINIB,MCL1,other,other,NaN,None
5,PD001319,IMATINIB,KIT,antagonist,negative,NaN,None
6,PD001319,IMATINIB,RET,inhibitor,negative,NaN,None
7,PD001319,IMATINIB,PDGFRA,antagonist,negative,NaN,None
8,PD001319,IMATINIB,ABL1,inhibitor,negative,NaN,None
9,PD001319,IMATINIB,PDGFRB,antagonist,negative,NaN,None


---
## 1b. Which set is the compound derived from?

Compounds can belong to multiple sets (commercial libraries, probe sets, drug sets, etc.). The `get_compound_sets()` function returns all set memberships with type labels.

In [5]:
# Source sets for JQ1
df_jq1_sets = get_compound_sets('(+)-JQ1')
print(f"JQ1 belongs to {len(df_jq1_sets)} sets")
df_jq1_sets

Raw SQL output (32 rows):
  pdid | compound_name | set_name | set_type | version
  --------------------------------------------------------------------------------
  PD000031 | (+)-JQ1 | AdooQ Bioactive Compound  | Commercial compound sets | 
  PD000031 | (+)-JQ1 | Axon Medchem Screening Li | Commercial compound sets | 
  PD000031 | (+)-JQ1 | Cayman Chemical Bioactive | Commercial compound sets | 
  PD000031 | (+)-JQ1 | Enamine Bioactive Compoun | Commercial compound sets | 
  PD000031 | (+)-JQ1 | MedChem Express Bioactive | Commercial compound sets | 
  PD000031 | (+)-JQ1 | Selleckchem Bioactive Com | Commercial compound sets | 
  PD000031 | (+)-JQ1 | TargetMol Bioactive Compo | Commercial compound sets | 
  PD000031 | (+)-JQ1 | Tocris Bioactive Compound | Commercial compound sets | 
  PD000031 | (+)-JQ1 | Tocriscreen Plus | Commercial compound sets | 
  PD000031 | (+)-JQ1 | DrugBank | Drug compound sets | 5.1.13
  PD000031 | (+)-JQ1 | DrugMAP | Drug compound sets | 
  PD000031 | (+)-

,pdid,compound_name,set_name,set_type,version
0,PD000031,(+)-JQ1,AdooQ Bioactive Compound Library,Commercial compound sets,
1,PD000031,(+)-JQ1,Axon Medchem Screening Library,Commercial compound sets,
2,PD000031,(+)-JQ1,Cayman Chemical Bioactives,Commercial compound sets,
3,PD000031,(+)-JQ1,Enamine Bioactive Compounds,Commercial compound sets,
4,PD000031,(+)-JQ1,MedChem Express Bioactive Compound Library,Commercial compound sets,
5,PD000031,(+)-JQ1,Selleckchem Bioactive Compound Library,Commercial compound sets,
6,PD000031,(+)-JQ1,TargetMol Bioactive Compound Library,Commercial compound sets,
7,PD000031,(+)-JQ1,Tocris Bioactive Compound Library,Commercial compound sets,
8,PD000031,(+)-JQ1,Tocriscreen Plus,Commercial compound sets,
9,PD000031,(+)-JQ1,DrugBank,Drug compound sets,5.1.13


In [6]:
# Source sets for Imatinib
df_imatinib_sets = get_compound_sets('IMATINIB')
print(f"Imatinib belongs to {len(df_imatinib_sets)} sets")
df_imatinib_sets

Raw SQL output (42 rows):
  pdid | compound_name | set_name | set_type | version
  --------------------------------------------------------------------------------
  PD001319 | IMATINIB | AdooQ Bioactive Compound  | Commercial compound sets | 
  PD001319 | IMATINIB | Enamine BioReference Comp | Commercial compound sets | 
  PD001319 | IMATINIB | Enamine Bioactive Compoun | Commercial compound sets | 
  PD001319 | IMATINIB | MedChem Express Bioactive | Commercial compound sets | 
  PD001319 | IMATINIB | Prestwick Chemical Librar | Commercial compound sets | 18
  PD001319 | IMATINIB | Selleckchem Bioactive Com | Commercial compound sets | 
  PD001319 | IMATINIB | TargetMol Bioactive Compo | Commercial compound sets | 
  PD001319 | IMATINIB | The Spectrum Collection | Commercial compound sets | 
  PD001319 | IMATINIB | CeMM library of unique dr | Drug compound sets | 
  PD001319 | IMATINIB | ChEMBL Approved Drugs | Drug compound sets | 36
  PD001319 | IMATINIB | ChEMBL Drugs | Drug compou

,pdid,compound_name,set_name,set_type,version
0,PD001319,IMATINIB,AdooQ Bioactive Compound Library,Commercial compound sets,
1,PD001319,IMATINIB,Enamine BioReference Compounds,Commercial compound sets,
2,PD001319,IMATINIB,Enamine Bioactive Compounds,Commercial compound sets,
3,PD001319,IMATINIB,MedChem Express Bioactive Compound Library,Commercial compound sets,
4,PD001319,IMATINIB,Prestwick Chemical Library,Commercial compound sets,18
5,PD001319,IMATINIB,Selleckchem Bioactive Compound Library,Commercial compound sets,
6,PD001319,IMATINIB,TargetMol Bioactive Compound Library,Commercial compound sets,
7,PD001319,IMATINIB,The Spectrum Collection,Commercial compound sets,
8,PD001319,IMATINIB,CeMM library of unique drugs (CLOUD),Drug compound sets,
9,PD001319,IMATINIB,ChEMBL Approved Drugs,Drug compound sets,36


---
## 1c. What is the compound's main target, and how selective is it?

### Primary target

The `primary_target` flag in `compoundaction` identifies the intended target. The `get_primary_target()` function retrieves it.

In [7]:
# Primary targets
df_jq1_primary = get_primary_target('(+)-JQ1')
df_imatinib_primary = get_primary_target('IMATINIB')

print("JQ1 primary targets:")
print(df_jq1_primary[['compound_name', 'target_gene', 'action']].to_string(index=False))
print()
print("Imatinib primary targets:")
print(df_imatinib_primary[['compound_name', 'target_gene', 'action']].to_string(index=False))

Raw SQL output: 0 rows

Raw SQL output (1 rows):
  pdid | compound_name | target_gene | action | primary_target | drug_target
  --------------------------------------------------------------------------------
  PD001319 | IMATINIB | ABL1 | inhibitor | 1 | NULL

JQ1 primary targets:
Empty DataFrame
Columns: [compound_name, target_gene, action]
Index: []

Imatinib primary targets:
compound_name target_gene    action
     IMATINIB        ABL1 inhibitor


### Pre-computed selectivity metrics

The `compoundbasetargetcriteria` table stores curated potency, selectivity, and family selectivity scores. Use `get_selectivity()` to retrieve them.

In [8]:
# Pre-computed selectivity for JQ1
df_jq1_sel = get_selectivity('(+)-JQ1')
df_jq1_sel

Raw SQL output (4 rows):
  pdid | compound_name | target_gene | potency | selectivity | selectivity_score | family_selectivity | cell_potency | potency_selectivity_synergy
  --------------------------------------------------------------------------------
  PD000031 | (+)-JQ1 | BRDT | 7.96 | 0.66 | 0.0 | 0.62 | 6.87 | 0
  PD000031 | (+)-JQ1 | BRD4 | 7.3 | NULL | 0.0 | 0.62 | 7.2 | 0
  PD000031 | (+)-JQ1 | BRD3 | 7.16 | NULL | 0.0 | 0.62 | 7.93 | 0
  PD000031 | (+)-JQ1 | BRD2 | 7.16 | NULL | 0.0 | 0.62 | 7.88 | 0



,pdid,compound_name,target_gene,potency,selectivity,selectivity_score,family_selectivity,cell_potency,potency_selectivity_synergy
0,PD000031,(+)-JQ1,BRDT,7.96,0.66,0.0,0.62,6.87,0
1,PD000031,(+)-JQ1,BRD4,7.30,NaN,0.0,0.62,7.20,0
2,PD000031,(+)-JQ1,BRD3,7.16,NaN,0.0,0.62,7.93,0
3,PD000031,(+)-JQ1,BRD2,7.16,NaN,0.0,0.62,7.88,0


In [9]:
# Pre-computed selectivity for Imatinib
df_imatinib_sel = get_selectivity('IMATINIB')
df_imatinib_sel

Raw SQL output (4 rows):
  pdid | compound_name | target_gene | potency | selectivity | selectivity_score | family_selectivity | cell_potency | potency_selectivity_synergy
  --------------------------------------------------------------------------------
  PD001319 | IMATINIB | ABL1,BCR | 7.0 | NULL | 0.0 | NULL | 6.26 | 0
  PD001319 | IMATINIB | ABL1 | 6.7 | NULL | 0.0 | NULL | 6.52 | 0
  PD001319 | IMATINIB | KIT | 7.0 | NULL | 0.0 | NULL | 6.43 | 0
  PD001319 | IMATINIB | PDGFRB | 6.96 | NULL | 0.0 | NULL | 7.06 | 0



,pdid,compound_name,target_gene,potency,selectivity,selectivity_score,family_selectivity,cell_potency,potency_selectivity_synergy
0,PD001319,IMATINIB,"ABL1,BCR",7.00,None,0.0,None,6.26,0
1,PD001319,IMATINIB,ABL1,6.70,None,0.0,None,6.52,0
2,PD001319,IMATINIB,KIT,7.00,None,0.0,None,6.43,0
3,PD001319,IMATINIB,PDGFRB,6.96,None,0.0,None,7.06,0


### Compute your own S-score

The **S-score** is the fraction of targets with potency ≥ 6.0 (≈ 1 µM). A lower S-score means a more selective compound. The `compute_s_score()` function calculates it from raw activity data.

In [10]:
# S-scores for both compounds
for name in ['(+)-JQ1', 'IMATINIB']:
    result = compute_s_score(name, threshold=6.0)
    print(f"{result['compound_name']}: "
          f"{result['n_targets']} targets, {result['n_hit']} with potency >= 6.0, "
          f"S-score = {result['s_score']:.3f}")
    print(f"  Most potent target: {result['top_target']}")
    print()

Raw SQL output (11 rows):
  pdid | compound_name | target_gene | target_name | best_potency | activity_types | value_types | n_measurements
  --------------------------------------------------------------------------------
  PD000031 | (+)-JQ1 | BRD4 | Bromodomain-containing pr | 9.0 | pKd,pIC50,pKi,pEC50 | median,= | 290
  PD000031 | (+)-JQ1 | BRD2 | Bromodomain-containing pr | 8.41 | pKd,pKi,pIC50 | = | 53
  PD000031 | (+)-JQ1 | BRD3 | Bromodomain-containing pr | 8.4 | pKd,pKi,pIC50 | = | 42
  PD000031 | (+)-JQ1 | BRDT | Bromodomain testis-specif | 8.4 | pIC50,pKd | median,= | 34
  PD000031 | (+)-JQ1 | CCL2 | C-C motif chemokine 2 | 8.2 | pIC50 | = | 2
  PD000031 | (+)-JQ1 | HDAC1,HDAC10,HDAC11,HDAC2 | Histone deacetylase | 6.54 | pIC50 | = | 1
  PD000031 | (+)-JQ1 | DNER | Delta and Notch-like epid | 6.3 | pIC50 | = | 1
  PD000031 | (+)-JQ1 | CREBBP | CREB-binding protein | 5.02 | pIC50,pKd | = | 3
  PD000031 | (+)-JQ1 | JAK2 | Tyrosine-protein kinase J | 5.0 | pIC50 | = | 1
  PD000

(+)-JQ1: 11 targets, 7 with potency >= 6.0, S-score = 0.636
  Most potent target: BRD4 (pKd=9.0)



Raw SQL output (89 rows):
  pdid | compound_name | target_gene | target_name | best_potency | activity_types | value_types | n_measurements
  --------------------------------------------------------------------------------
  PD001319 | IMATINIB | ERBB2 | Receptor tyrosine-protein | 10.22 | pIC50 | = | 1
  PD001319 | IMATINIB | EGFR | Epidermal growth factor r | 9.96 | pKd,pIC50 | = | 3
  PD001319 | IMATINIB | DDR1 | Epithelial discoidin doma | 9.15 | pIC50,pKd | median,= | 9
  PD001319 | IMATINIB | ABL1 | Tyrosine-protein kinase A | 9.0 | pIC50,pKd,pKi,pPotency | median,= | 111
  PD001319 | IMATINIB | ABL1,BCR | Bcr/Abl fusion protein | 8.96 | pIC50,pKi,pEC50 | = | 52
  PD001319 | IMATINIB | PDGFRA | Platelet-derived growth f | 8.7 | pIC50,pEC50,pKd,pKi | = | 13
  PD001319 | IMATINIB | ABL2 | Tyrosine-protein kinase A | 8.22 | pKd,pIC50 | = | 14
  PD001319 | IMATINIB | BCR | Breakpoint cluster region | 8.15 | pKd | = | 1
  PD001319 | IMATINIB | CSF1R | Macrophage colony-stimula | 8.0 |

IMATINIB: 89 targets, 41 with potency >= 6.0, S-score = 0.461
  Most potent target: ERBB2 (pIC50=10.22)



---
## 1d. Visualization: Potency profile across all targets

The `plot_potency_profile()` function creates a horizontal bar chart showing the selectivity gap between primary targets and off-targets.

Each bar is annotated with the **exact measurement type** that produced the value (e.g. `pKd=9.0` or `pIC50=8.4`). This matters because `MAX(activity_value)` may pick a binding affinity (pKd) for one target and a functional inhibition (pIC50) for another — they're on the same -log₁₀(M) scale but measure different things.

Pass `hue_by_type=True` with a detailed DataFrame to draw **grouped bars** — one bar per activity type per target — so you can see all measurement types side by side.

In [11]:
# JQ1 potency profile — each bar annotated with measurement type
# Blue = primary BET bromodomain targets, grey = off-targets
plot_potency_profile(
    df_jq1_targets,
    compound_name='(+)-JQ1',
    primary_genes={'BRD2', 'BRD3', 'BRD4', 'BRDT'},
    threshold=6.0,
    save_path='/mnt/results/notebooks/fig_us1_jq1_potency_profile.png'
)

In [12]:
# JQ1 potency grouped by measurement type — one bar per activity type per target
# Shows that BRD4's best value is a pKd (binding), while BRD2/3/DT are pIC50 (functional)
df_jq1_detailed = get_compound_targets_detailed('(+)-JQ1')
plot_potency_profile(
    df_jq1_targets,
    compound_name='(+)-JQ1',
    threshold=6.0,
    hue_by_type=True,
    detailed_df=df_jq1_detailed,
    save_path='/mnt/results/notebooks/fig_us1_jq1_potency_by_type.png'
)

### Imatinib potency profile

Imatinib has 89 high-quality targets (conf=1, exact measurements only) — we show the top 25. Each bar is annotated with the measurement type. Targets flagged with `(!)` have contradictory binding vs functional data (see data quality note in summary).

In [13]:
# Imatinib top 25 targets — each bar annotated with measurement type
df_imatinib_top = df_imatinib_targets.head(25)
plot_potency_profile(
    df_imatinib_top,
    compound_name='IMATINIB (top 25 targets)',
    primary_genes={'ABL1'},
    threshold=6.0,
    save_path='/mnt/results/notebooks/fig_us1_imatinib_potency_profile.png'
)

### Imatinib: grouped by measurement type (top 15)

For the top 15 targets, show all measurement types side by side. This reveals where binding affinity (pKd) and functional inhibition (pIC50) diverge.

In [14]:
# Imatinib top 15 targets — grouped bars by activity type
df_imatinib_detailed = get_compound_targets_detailed('IMATINIB')
top15_genes = df_imatinib_targets.head(15)['target_gene'].tolist()
df_imatinib_det_top = df_imatinib_detailed[df_imatinib_detailed['target_gene'].isin(top15_genes)]
plot_potency_profile(
    df_imatinib_targets.head(15),
    compound_name='IMATINIB (top 15 targets)',
    threshold=6.0,
    hue_by_type=True,
    detailed_df=df_imatinib_det_top,
    save_path='/mnt/results/notebooks/fig_us1_imatinib_potency_by_type.png'
)

---
## Summary

| Metric | JQ1 | Imatinib |
|--------|-----|----------|
| PD ID | PD000031 | PD001319 |
| Total targets (high-quality) | 11 | 89 |
| Primary target | BRD4 (BET bromodomain) | ABL1 (kinase) |
| Best potency | pKd = 9.0 (BRD4) | pIC50 = 10.22 (ERBB2, flagged) |
| S-score (threshold 6.0) | 0.636 | 0.461 |
| Source sets | 32 | — |

**Key insight:** JQ1 is a selective BET bromodomain probe (S-score 0.64), while Imatinib is a polypharmacological kinase inhibitor (S-score 0.46) — consistent with its known multi-target profile (ABL1, KIT, PDGFR, DDR1/2).

**Data quality filtering:** By default, `get_compound_targets()` applies three filters:
1. **Log-scale only** — excludes percentage-scale measurements (Inhibition %, Dmax %)
2. **Exact measurements only** — excludes screening negatives (`>` values like "IC50 > 30 µM") that indicate the compound was tested but found inactive above a threshold
3. **Confidence = 1** — keeps only directly measured values, excluding derived/converted values (confidence = 2)

This reduces Imatinib from 432 raw targets to 89 high-quality targets. The 343 excluded targets are mostly screening panel negatives (tested at 10 µM, no activity detected).

**Contradiction flagging:** The `contradiction_flag` column marks targets where binding (pKd) and functional (pIC50) measurements disagree by >2 log units. For Imatinib, ERBB2 and EGFR are flagged: both have pKd > 5.0 (weak binding, >10 µM) but pIC50 ~10 (ultra-potent, <1 nM) — almost certainly ChEMBL annotation errors. ABL1 is NOT flagged because it has exact pKd = 9.0 (strong binding) consistent with its known potent inhibition.

**Note on activity values:** All potency values are -log₁₀(M) on a log scale (pIC50, pKd, pKi, pEC50). A value of 6.0 = 1 µM, 9.0 = 1 nM.

## Reusable functions used

| Function | Purpose |
|----------|---------|
| `get_compound_targets(name)` | All targets with best potency + measurement type (quality-filtered) |
| `get_compound_targets_detailed(name)` | Per-target per-activity-type breakdown |
| `get_compound_actions(name)` | Curated mechanism annotations |
| `get_compound_sets(name)` | Source compound sets |
| `get_primary_target(name)` | Primary target flag |
| `get_selectivity(name)` | Pre-computed selectivity metrics |
| `compute_s_score(name, threshold)` | Custom selectivity score |
| `plot_potency_profile(df, ...)` | Potency bar chart with type labels + contradiction flags |
| `plot_potency_profile(..., hue_by_type=True)` | Grouped bars by activity type |